# Klassifikation mit verschiedenen Leveln der Personalisierung

1. Subject-independent
2. Subject Baseline Norm - t0 als baseline; t0 mittel wird von allen samples abgezogen
3. Mit Kalibrierungs Samples - t0-t5; 2,4,6 samples pro stufe mit ins training
4. Nur auf Daten einer Person - cv within subject

In [ ]:
from pathlib import Path
import numpy as np
from scripts.myml import loso_binary_nested_cv, loso_multiclass_nested_cv, loso_binary_calibrated_nested_cv
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from scripts.feature_engeneering import subject_baseline_normalization
from xgboost import XGBClassifier

### Daten Laden

In [4]:
DATASET_PATH = Path("dataset/np-dataset")
X = np.load(DATASET_PATH / "X.npy")
X_catch22 = np.load(DATASET_PATH / "X_catch22.npy")
X_catch22_feature_names = np.load(DATASET_PATH / "feature_names.npy")
y_heater = np.load(DATASET_PATH / "y_heater.npy")
subjects = np.load(DATASET_PATH / "subjects.npy")
y = np.argmax(y_heater, axis=1)

In [5]:
print(f"X shape : {X_catch22.shape}")
print(f"y shape : {y.shape}   classes: {np.unique(y).tolist()}")
print(f"subjects : {np.unique(subjects).shape[0]} unique subjects")
print(f"Class counts : { {c: int((y==c).sum()) for c in range(6)} }")

X shape : (2495, 174)
y shape : (2495,)   classes: [0, 1, 2, 3, 4, 5]
subjects : 52 unique subjects
Class counts : {0: 416, 1: 416, 2: 416, 3: 415, 4: 416, 5: 416}


### Hyperparameter Grid

In [ ]:
# Random Forest
PARAM_GRID_RF = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
}

PARAM_GRID_XGB = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 6, 10],
    'learning_rate': [0.01, 0.1, 0.2],
}

INNER_N_SPLITS = 5
RANDOM_STATE = 42

In [8]:
# without normalization
X_catch22 = X_catch22.copy()

# with subject-specific normalization
baseline_class = 0 # samples with no stimulus 
X_catch22_normalized = subject_baseline_normalization(X_catch22, y, subjects, baseline_class)

print(np.nanmean(X_catch22))
print(np.nanmean(X_catch22_normalized))

40.75209365022363
1.2530141390178788


# Multiclass t0-t5 - Random Forest Classifier

## 1. Subject-norm-baseline

Frage hierzu - normalerweise zählt es als data leakage, wenn man erst alles Normalisiert und danach Train/ test splits macht. Hier nutze ich jedoch nur t0 segmente um eine Null baseline zu schaffen, die im klinischen einsatz auch verfügbar wären. Daher okay? - da es als kalibrierung dient?

# 2. Mit Kalibrierung - Random Forest Classifier

# Mutliclass t0-t5 - XGBoost

## 1. Subject-norm-baseline

In [ ]:
pers_comparison = [X_catch22, X_catch22_normalized]

for X_comp in pers_comparison:
    
    print(f"\nPersonalization: {'Baseline Normalized' if X_comp is X_catch22_normalized else 'No Normalization'}")

    # Define model
    xgb_classifier = XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss')

    print(f"Training model: {xgb_classifier}...")
    metrics = loso_multiclass_nested_cv(
        X_comp, 
        y, 
        subjects, 
        xgb_classifier, 
        PARAM_GRID_XGB, 
        'classifier'
        )
    
    print(f"    Accuracy : {metrics['accuracy']:.3f} ± {metrics['accuracy_std']:.3f}")
    print(f"    F1       : {metrics['f1']:.3f} ± {metrics['f1_std']:.3f}")
    print(f"    AUC      : {metrics['auc']:.3f} ± {metrics['auc_std']:.3f}")
    print(f"    MAE      : {metrics['mae']:.3f} ± {metrics['mae_std']:.3f}")
    print(f"    RMSE      : {metrics['rmse']:.3f} ± {metrics['rmse_std']:.3f}")

## 